In [ ]:
# Paths come from the memtune package (see docs/INSTALL.md): no machine-specific locations.
#   data   -> data/lbfgs_runs.csv   (override: MEMTUNE_DATA_DIR)
#   models -> models/              (override: MEMTUNE_MODELS_DIR)
#   output -> results/             (override: MEMTUNE_RESULTS_DIR)
from memtune import paths
(paths.results_dir() / "retrained_models").mkdir(parents=True, exist_ok=True)  # notebooks never overwrite models/


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
error_cols = ["error_problem", "error_solver", "error_type"]
raw_df = pd.read_csv(str(paths.dataset_path()))
raw_df

In [ ]:
df = pd.read_csv(str(paths.data_dir() / "final.csv"))
df.columns

In [ ]:
data = pd.read_csv(str(paths.dataset_path()))
data[data["status"].notna()]
data["status"].value_counts()

In [ ]:
list(data.columns)

In [ ]:
meta_df = pd.read_csv(str(paths.data_dir() / "problem_metadata.csv"))
meta_df.head(10)
len(list(meta_df.columns))

In [ ]:
solver_df = pd.read_csv(str(paths.data_dir() / "solver_runtimes.csv"))
solver_df.head(10)
len(list(solver_df.columns))

In [ ]:
data["name"]

## Get a sense of how many problems are left unsolved

In [ ]:
bins = [0, 95, 100, 1_000, 10_000, 100_000, 500_000]
data["nvar_bin"] = pd.cut(data["nvar"], bins=bins)
bin_counts = data["nvar_bin"].value_counts().sort_index()
print(bin_counts)

## Timeout is the main cause of error. We will assign it with the maxtime

In [ ]:
# g = error_df.groupby(["name", "nvar"])

# summary = g.agg(
#     mem_tried=("mem", "nunique"),
#     n_failed=("error_solver", lambda s: s.notna().sum()),
#     n_timeout=("error_solver", lambda s: s.astype(str).str.contains("timeout|max_time|time limit", case=False, na=False).sum()),
# ).reset_index()

# summary["all_failed"] = summary["n_failed"] == summary["mem_tried"]
# summary["all_timeout"] = summary["n_timeout"] == summary["mem_tried"]

# # summary[summary["all_failed"]]
# summary[summary["all_timeout"]].sum()

In [ ]:
# error_df = pd.read_csv("error.csv")
# error_df[error_cols]
# error_df["error_type"].value_counts()

In [ ]:
data["extract_duration_solver"].describe()

In [ ]:
data["extract_duration_problem"].describe()

In [ ]:
data["stats_elapsed_time"].describe()

In [ ]:
data["eval_duration_solver"].describe() # time to instantiate the problem

In [ ]:
data["eval_duration_problem"].describe() # time to instantiate before problem and extract features

In [ ]:
col_lst = [column for column in data.columns]
# for col in col_lst:
#     print(col)
col_lst

In [ ]:
# success_counts = data.groupby("mem")["solved"].sum().sort_values(ascending=False)
# success_counts

In [ ]:
status_counts = data["status"].value_counts()

plt.figure()
plt.bar(status_counts.index.astype(str), status_counts.values)
plt.xlabel("Status")
plt.ylabel("Count")
plt.title("Solver Status Distribution")
plt.show()

In [ ]:
data["stats_elapsed_time"].describe()

In [ ]:
data["is_constant (ExprTree)"].dropna().value_counts()

In [ ]:
data["is_linear (ExprTree)"].value_counts()

In [ ]:
x = data["highest_degree (ExprTree)"].dropna()
counts = x.value_counts().sort_index()
plt.figure()
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Highest Degree")
plt.ylabel("Frequency")
plt.show()

In [ ]:
x = data["tree_length"].dropna()
plt.figure()
plt.title("Distribution of Highest Degree (ExprTree) raw data")
plt.hist(x, bins="scott")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

x = data["tree_length"].dropna()
x = x[x <= 25000]
plt.figure()
plt.title("Distribution of Highest Degree (ExprTree) raw data [0,25000]")
plt.hist(x, bins="scott")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

ax = sns.histplot(np.log10(x), bins=50)
ax.set_title("Histogram of log-scaled tree_length")


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Match academic font styles (Serif layout)
plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": False, 
    "font.size": 11
})

# 1. Isolate and log-transform the data
# Using log10(x + 1) to safely handle potential zeros
x_log = np.log10(data["tree_length"].dropna() + 1)

# 2. Setup the plot figure
plt.figure(figsize=(7, 4.5), dpi=150)

# 3. Plot using the matching lighter blue color from the histograms
sns.boxplot(y=x_log, color="#5b92e5", width=0.4, flierprops={"marker": "o", "alpha": 0.5})

# 4. Refine titles and math-formatted axis labels
plt.title(r"Distribution of $\log_{10}$(\texttt{tree\_length})", fontsize=12, pad=12)
plt.ylabel(r"$\log_{10}$(\texttt{tree\_length} + 1)", fontsize=11, labelpad=8)

# 5. Add a subtle background grid
plt.grid(axis='y', linestyle=':', linewidth=0.5, color='#b0b0b0', alpha=0.7)
plt.gca().set_axisbelow(True)

# 6. Clean up the frame spines
for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(y=x_log, width=0.4)
plt.title("Distribution of log-scaled tree_length")

In [ ]:
x = data["tree_depth"].dropna()
counts = x.value_counts().sort_index()
plt.figure()
plt.title("Distribution of tree_depth")
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Tree Depth")
plt.ylabel("Frequency")
plt.show()

In [ ]:
x = data["count_plus_minus"].dropna()
plt.figure()
plt.title("Distribution of count_plus_minus raw data")
plt.hist(x, bins="scott")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

x = data["count_plus_minus"].dropna()
x = x[x <= 100000]
plt.figure()
plt.title("Distribution of count_plus_minus raw data [0,100000]")
plt.hist(x, bins="scott")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

ax = sns.histplot(np.log10(x), bins=40, kde=True)
ax.set_title("Log-scaled Distribution of count_plus_minus raw data")

In [ ]:
x = data["count_trigonometric_function"].dropna()
count = x.value_counts().sort_index()
plt.title("Distribution of count_trignometric_function")
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
x = data["count_exponential_function"].dropna()
count = x.value_counts().sort_index()
plt.title("Distribution of count_exponential_function")
plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
x = data["count_op_function"]
x.describe()

sns.histplot(x=np.log10(x))
plt.xlabel("log_10(count_op_function)")
plt.ylabel("counts")
plt.title("Histogram of log-scaled count_op_function in the dataset")
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Match academic font styles (Serif layout)
plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": False, 
    "font.size": 10  # Slightly smaller font to balance the narrower width
})

# 1. Isolate and log-transform the data
x_log = np.log10(data["count_op_function"].dropna())

# 2. Setup a narrower plot figure
plt.figure(figsize=(5.5, 4.0), dpi=150)

# 3. Plot using the cohesive steelblue color with no edge borders
sns.histplot(x=x_log, bins=40, color="steelblue", edgecolor="none", alpha=0.85)

# 4. Refine titles and clean plain-text labels using log_10()
plt.title("Histogram of log-scaled count_op_function", fontsize=11, pad=10)
plt.xlabel("log_10(count_op_function)", fontsize=10, labelpad=6)
plt.ylabel("Frequency", fontsize=10, labelpad=6)

# 5. Add a subtle background grid
plt.grid(axis='y', linestyle=':', linewidth=0.5, color='#b0b0b0', alpha=0.7)
plt.gca().set_axisbelow(True)

# 6. Clean up the frame spines
for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
x = data["neval_obj"].dropna()

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(neval_obj)")
plt.title("Distribution of log-scaled number of evaluations of objective function")
plt.show()

In [ ]:
x = data["neval_grad"].dropna()

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(neval_grad)")
plt.title("Distribution of log-scaled neval_grad")
plt.show()

In [ ]:
x = data["time_obj"].dropna()

sns.boxplot(y=np.log10(x))
plt.ylabel("neval_obj")
plt.show()

In [ ]:
x = data["time_grad"].dropna()

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(time_grad)")
plt.title("Distribution of log-scaled time_grad")
plt.show()

In [ ]:
x = data["alloc_grad"].dropna()
x = x[x > 0]

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(alloc_grad)")
plt.show()

In [ ]:
x = data["alloc_grad"].dropna()
x = x[x > 0]

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(alloc_grad)")
plt.show()

In [ ]:
x = data["eval_duration_problem"].dropna()

sns.boxplot(y=x)
plt.ylabel("eval_duration_problem")
plt.show()

In [ ]:
x = data["eval_duration_solver"].dropna()
x = x[x > 0]

sns.boxplot(y=x)
plt.ylabel("eval_duration_solver")
plt.show()

In [ ]:
x = data["stats_elapsed_time"]
# x = x[x>0]

sns.boxplot(y=np.log10(x))
plt.ylabel("log10(stats_elapsed_time)")
plt.title("Distribution of log-scaled solver_runtime")
plt.show()

data["stats_elapsed_time"].describe()

In [ ]:
x = data["total_alloc"]
x = x[x>0]
x.describe()

sns.boxplot(y=np.log10(x))
plt.ylabel("total_alloc")
plt.title("Distribution of log-scaled total_alloc")
plt.show()

In [ ]:
data["stats_elapsed_time"].describe()

In [ ]:
status_table = (
    data["status"]
    .dropna()
    .astype(str)                  
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)
status_table["proportion"] = status_table["count"] / status_table["count"].sum()
status_table

In [ ]:
plt.figure()
plt.bar(status_table["status"], status_table["count"])
plt.yscale("log")
plt.xlabel("Status")
plt.ylabel("Count (log scale)")
plt.title("Solver Status Distribution (log scale)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
x = data["timed_bytes"]
x.describe()

sns.histplot(x=np.log10(x))
plt.xlabel("log10(memory_usage)")
plt.ylabel("counts")
plt.title("Histogram of log-scaled memory_usage in the dataset")
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Match academic font styles (Serif layout)
plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": False, 
    "font.size": 10
})

x = data["timed_bytes"]
x.describe()

# Setup a narrower, cleaner plot figure size
plt.figure(figsize=(5.5, 4.0), dpi=150)

# Plot with cohesive steelblue color, removing harsh dark bar edges
sns.histplot(x=np.log10(x), bins=50, color="steelblue", edgecolor="none", alpha=0.85)

# Keep your original labels and titles completely untouched
plt.xlabel("log10(memory_usage)", fontsize=10, labelpad=6)
plt.ylabel("counts", fontsize=10, labelpad=6)
plt.title("Histogram of log-scaled memory_usage in the dataset", fontsize=11, pad=10)

# Add a subtle background grid line system
plt.grid(axis='y', linestyle=':', linewidth=0.5, color='#b0b0b0', alpha=0.7)
plt.gca().set_axisbelow(True)

# Clean up top and right frame borders
for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
x = data["timed_bytes"]
x.describe()

sns.boxplot(y=np.log1p(x))
plt.ylabel("counts")
plt.title("Histogram of log-scaled memory usage in the dataset")
plt.show()

In [ ]:
x = data["neval_obj"]
x.describe()

sns.histplot(x=np.log1p(x))
plt.ylabel("counts")
plt.title("Histogram of log-scaled memory usage in the dataset")
plt.show()

In [ ]:
x = data["neval_grad"]
x.describe()

sns.histplot(x=np.log1p(x))
plt.ylabel("counts")
plt.title("Histogram of log-scaled memory usage in the dataset")
plt.show()

In [ ]:
x = data["stats_elapsed_time"]
x.describe()

sns.histplot(x=np.log1p(x))
plt.ylabel("counts")
plt.title("Histogram of log-scaled memory usage in the dataset")
plt.show()

In [ ]:
x = data["timed_time"]
x.describe()

sns.boxplot(y=np.log10(x))
plt.ylabel("timed_time")
plt.show()

In [ ]:
x = data["timed_gctime"]
x.describe()

sns.boxplot(y=np.log10(x + 0.00000000000001))
plt.ylabel("timed_gctime")
plt.show()

In [ ]:
x = data["stats_elapsed_time"]
x.describe()

sns.boxplot(y=np.log10(x))
plt.ylabel("stats_elapsed_time")
plt.show()

In [ ]:
x = data["nvar"]
x.describe()

sns.boxplot(y=np.log10(x))
plt.xlabel("Distribution of problem size")
plt.show()

In [ ]:
data["nvar"].value_counts()

In [ ]:
intervals = [100, 1000, 10000, 100000]
data["nvar"] = data["nvar"].apply(lambda x: min(intervals, key=lambda i: abs(i - x)))

In [ ]:
data["nvar"].value_counts()

In [ ]:
counts = data["nvar"].value_counts().sort_index()

plt.bar(counts.index.astype(str), counts.values)
plt.xlabel("Number of variables (nvar)")
plt.ylabel("Count")
plt.title("Dataset composition by problem size (nvar)")
plt.show()